# Retail Demand Forecasting — Model

Compact model notebook: creates lag/calendar features, uses a chronological 90-day holdout, compares XGBoost against a 7-day-lag baseline, and saves the trained model.

**Evaluation:** one-day-ahead rolling predictions using observed previous sales (not a full 90-day forecast).

## 1. Load retail + weather + trend data\nPlace `weather_demo.csv` and `social_trends_demo.csv` in `data/`.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd, joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, OUT = ROOT/"data", ROOT/"outputs"
OUT.mkdir(exist_ok=True)

df = pd.read_csv(DATA/"retail_merged.csv", parse_dates=["date"])
weather = pd.read_csv(DATA/"weather_demo.csv", parse_dates=["date"])
trends = pd.read_csv(DATA/"social_trends_demo.csv", parse_dates=["date"])

df = df.merge(weather.drop(columns="data_status"), on=["date","city"], how="left")
df = df.merge(trends.drop(columns="data_status"), on=["date","family"], how="left")
df = df.sort_values(["store_nbr","family","date"]).reset_index(drop=True)
print("Data:", df.shape, "| weather missing:", df.temperature_2m_mean.isna().sum(),
      "| trend missing:", df.trend_index.isna().sum())
display(df.head())

Data: (3000888, 20) | weather missing: 111144 | trend missing: 2455272


,id,date,store_nbr,family,sales,onpromotion,promotion,day_name,city,state,type,cluster,transactions,dcoilwtico,is_holiday,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,trend_index
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,No promotion,Tuesday,Quito,Pichincha,D,13,NaN,NaN,1,16.5,19.7,13.4,3.9,NaN
1,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,No promotion,Wednesday,Quito,Pichincha,D,13,2111.0,93.14,0,17.1,20.3,14.0,4.8,NaN
2,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,No promotion,Thursday,Quito,Pichincha,D,13,1833.0,92.97,0,20.2,23.4,17.1,3.5,NaN
3,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,No promotion,Friday,Quito,Pichincha,D,13,1863.0,93.12,0,18.6,21.8,15.5,5.8,NaN
4,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,No promotion,Saturday,Quito,Pichincha,D,13,1509.0,NaN,1,17.5,20.7,14.4,3.3,NaN


**Demo note:** weather and trend files are synthetic placeholders; replace with real observations before reporting research results.

## 2. Feature engineering

In [2]:
# Calendar features
df["month"] = df.date.dt.month
df["dayofweek"] = df.date.dt.dayofweek
df["weekend"] = (df.dayofweek >= 5).astype("int8")

# Past sales only: shift before rolling to avoid target leakage
g = df.groupby(["store_nbr", "family"], sort=False)["sales"]
for lag in (7, 14, 28):
    df[f"lag_{lag}"] = g.shift(lag)
df["roll_7"] = g.transform(lambda s: s.shift(1).rolling(7).mean())
df["roll_28"] = g.transform(lambda s: s.shift(1).rolling(28).mean())

df = df.dropna(subset=["lag_7", "lag_14", "lag_28", "roll_7", "roll_28"]).copy()
print("Feature data:", df.shape)

Feature data: (2950992, 28)


## 3. Time-based split and encoding

In [3]:
cutoff = df.date.max() - pd.Timedelta(days=90)
train, valid = df[df.date < cutoff], df[df.date >= cutoff]

target = "sales"
features = [c for c in df.columns if c not in ["sales", "date", "day_name", "promotion"]]

# Encode categorical columns using training data only
from sklearn.preprocessing import OrdinalEncoder
cat_cols = train[features].select_dtypes(include=["object", "category"]).columns.tolist()
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

X_train, X_valid = train[features].copy(), valid[features].copy()
if cat_cols:
    X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols].astype(str))
    X_valid[cat_cols] = encoder.transform(X_valid[cat_cols].astype(str))

X_train, X_valid = X_train.astype("float32"), X_valid.astype("float32")
y_train, y_valid = train[target], valid[target]
print("Train:", X_train.shape, "| Validation:", X_valid.shape)
print("Validation:", valid.date.min().date(), "to", valid.date.max().date())

C:\Users\darks\AppData\Local\Temp\ipykernel_1296\3399490721.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train[features].select_dtypes(include=["object", "category"]).columns.tolist()


Train: (2788830, 24) | Validation: (162162, 24)
Validation: 2017-05-17 to 2017-08-15


## 4. Train XGBoost

In [4]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=350, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective="reg:squarederror", n_jobs=-1, random_state=42
)
model.fit(X_train, y_train)
pred = np.maximum(0, model.predict(X_valid))
print("Model trained.")

Model trained.


## 5. Evaluate model vs baseline

In [5]:
def metrics(actual, forecast):
    mae = mean_absolute_error(actual, forecast)
    rmse = np.sqrt(mean_squared_error(actual, forecast))
    wape = np.abs(actual - forecast).sum() / np.abs(actual).sum() * 100
    return {"MAE": round(mae, 3), "RMSE": round(rmse, 3), "WAPE_%": round(wape, 2)}

scores = pd.DataFrame({
    "7-day lag baseline": metrics(y_valid, valid["lag_7"]),
    "XGBoost": metrics(y_valid, pred)
}).T
display(scores)

,MAE,RMSE,WAPE_%
7-day lag baseline,87.572,335.421,18.25
XGBoost,59.853,229.932,12.47


## 6. Save predictions and model

In [6]:
results = valid[["date", "store_nbr", "family", "sales"]].copy()
results["prediction"] = pred
results.to_csv(OUT / "validation_predictions.csv", index=False)

joblib.dump({
    "model": model,
    "encoder": encoder,
    "categorical_columns": cat_cols,
    "features": features,
    "validation_cutoff": cutoff
}, OUT / "demand_model.joblib")

print("Saved outputs/demand_model.joblib and validation_predictions.csv")

Saved outputs/demand_model.joblib and validation_predictions.csv
